# 01 — Data Quality Check

This notebook profiles the source data, documents data quality issues, and produces a cleaned user table for downstream work.




## Objectives

- Confirm table sizes and core key relationships
- Get familiar with each dataset (shape, unique values, missingness, date ranges)
- Identify structural issues (duplicates, broken foreign keys)
- Identify temporal inconsistencies

In [11]:

from pathlib import Path
import sys
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().resolve().parents[0]
sys.path.append(str(PROJECT_ROOT / "src"))
SNAPSHOT_DATE = pd.Timestamp("2024-06-30")


import syskit_utils as su

DB_PATH = "saas_dataset.sqlite"
TABLES = su.load_tables(DB_PATH)


In [12]:
def dataset_familiarity_profile(tables: dict):
    table_rows = []
    variable_rows = []
    date_rows = []

    for table_name, df in tables.items():
        rows, cols = df.shape
        null_cells = int(df.isna().sum().sum())
        table_rows.append(
            {
                "table_name": table_name,
                "rows": int(rows),
                "columns": int(cols),
                "null_cells": null_cells,
                "null_pct": (null_cells / max(rows * cols, 1)) * 100,
            }
        )

        for col in df.columns:
            s = df[col]
            non_null = int(s.notna().sum())
            unique_non_null = int(s.nunique(dropna=True))
            variable_rows.append(
                {
                    "table_name": table_name,
                    "column_name": col,
                    "dtype": str(s.dtype),
                    "non_null": non_null,
                    "missing_pct": ((len(s) - non_null) / max(len(s), 1)) * 100,
                    "unique_non_null": unique_non_null,
                    "unique_ratio": unique_non_null / max(non_null, 1),
                }
            )

            col_name = str(col).lower()
            looks_temporal = any(token in col_name for token in ["date", "time", "_at"])
            if pd.api.types.is_datetime64_any_dtype(s):
                parsed = s
            elif looks_temporal:
                parsed = pd.to_datetime(s, errors="coerce")
            else:
                parsed = None

            if parsed is not None and parsed.notna().any():
                min_date = parsed.min()
                max_date = parsed.max()
                date_rows.append(
                    {
                        "table_name": table_name,
                        "column_name": col,
                        "min_date": min_date,
                        "max_date": max_date,
                        "span_days": int((max_date - min_date).days),
                        "non_null_dates": int(parsed.notna().sum()),
                    }
                )

    table_profile = pd.DataFrame(table_rows).sort_values("rows", ascending=False).reset_index(drop=True)
    variable_profile = pd.DataFrame(variable_rows).sort_values(
        ["table_name", "unique_non_null"], ascending=[True, False]
    ).reset_index(drop=True)
    date_profile = pd.DataFrame(date_rows).sort_values(
        ["table_name", "column_name"]
    ).reset_index(drop=True)

    return table_profile, variable_profile, date_profile

table_profile, variable_profile, date_profile = dataset_familiarity_profile(TABLES)

display(table_profile)
display(variable_profile)
display(date_profile)

,table_name,rows,columns,null_cells,null_pct
0,events,366890,8,33151,1.12946
1,crm_activities,3477,6,0,0.00000
2,users,2779,6,0,0.00000
3,tenants,500,7,0,0.00000
4,subscriptions,500,8,425,10.62500
5,crm_companies,500,8,0,0.00000


,table_name,column_name,dtype,non_null,missing_pct,unique_non_null,unique_ratio
0,crm_activities,activity_id,object,3477,0.000000,3477,1.000000
1,crm_activities,days_to_renewal,int64,3477,0.000000,509,0.146391
2,crm_activities,tenant_id,object,3477,0.000000,496,0.142652
3,crm_activities,activity_date,datetime64[ns],3477,0.000000,214,0.061547
4,crm_activities,activity_type,object,3477,0.000000,5,0.001438
5,crm_activities,outcome,object,3477,0.000000,4,0.001150
6,crm_companies,company_id,object,500,0.000000,500,1.000000
7,crm_companies,tenant_id,object,500,0.000000,500,1.000000
8,crm_companies,created_at,datetime64[ns],500,0.000000,287,0.574000
9,crm_companies,industry,object,500,0.000000,8,0.016000


,table_name,column_name,min_date,max_date,span_days,non_null_dates
0,crm_activities,activity_date,2024-01-02 00:00:00,2024-09-09 00:00:00,251,3477
1,crm_companies,created_at,2023-01-02 00:00:00,2024-01-02 00:00:00,365,500
2,events,event_date,2024-01-02 00:00:00,2024-06-29 00:00:00,179,366890
3,events,event_time,2024-01-02 07:00:05,2024-06-29 19:59:56,179,366890
4,subscriptions,churn_date,2024-01-04 00:00:00,2024-09-30 00:00:00,270,75
5,subscriptions,contract_start_date,2023-01-02 00:00:00,2024-01-02 00:00:00,365,500
6,subscriptions,renewal_date,2024-01-02 00:00:00,2025-01-01 00:00:00,365,500
7,users,last_seen_at,2024-01-02 00:00:00,2024-09-22 00:00:00,264,2779
8,users,registered_at,2023-01-04 00:00:00,2024-01-30 00:00:00,391,2779


Some date fields (`activity_date`, `churn_date`, `last_seen_at`) extend beyond the telemetry cutoff (2024-06-29). This is not inherently problematic, but it requires careful approach to avoid data leakage during modeling. The collection timestamp for non-event tables is also unclear.


In [13]:
def dq_summary(tables):
    tenants = tables["tenants"]
    subscriptions = tables["subscriptions"]
    users = tables["users"]
    events = tables["events"]
    crm_companies = tables["crm_companies"]
    crm_activities = tables["crm_activities"]

    event_user_bounds = (
        events.groupby("user_id")
        .agg(
            first_event=("event_date", "min"),
            last_event=("event_date", "max"),
            rows=("event_id", "count"),
            total_event_count=("event_count", "sum"),
        )
        .reset_index()
    )
    user_check = users.merge(event_user_bounds, on="user_id", how="left")
    user_check["event_before_registered"] = (
        user_check["first_event"].notna()
        & user_check["registered_at"].notna()
        & (user_check["first_event"] < user_check["registered_at"])
    )
    user_check["event_after_last_seen"] = (
        user_check["last_event"].notna()
        & user_check["last_seen_at"].notna()
        & (user_check["last_event"] > user_check["last_seen_at"])
    )
    user_check["registered_after_last_seen"] = (
        user_check["registered_at"].notna()
        & user_check["last_seen_at"].notna()
        & (user_check["registered_at"] > user_check["last_seen_at"])
    )

    event_tenant_bounds = (
        events.groupby("tenant_id")
        .agg(
            first_event_tenant=("event_date", "min"),
            last_event_tenant=("event_date", "max"),
        )
        .reset_index()
    )
    subscription_event_check = subscriptions.merge(event_tenant_bounds, on="tenant_id", how="left")
    subscription_event_check["events_after_churn_date"] = (
        subscription_event_check["last_event_tenant"].notna()
        & subscription_event_check["churn_date"].notna()
        & (subscription_event_check["last_event_tenant"] > subscription_event_check["churn_date"])
    )
    subscription_event_check["events_before_contract_start_date"] = (
        subscription_event_check["first_event_tenant"].notna()
        & subscription_event_check["contract_start_date"].notna()
        & (subscription_event_check["first_event_tenant"] < subscription_event_check["contract_start_date"])
    )
    subscription_event_check["events_after_renewal_date"] = (
        subscription_event_check["last_event_tenant"].notna()
        & subscription_event_check["renewal_date"].notna()
        & (subscription_event_check["last_event_tenant"] > subscription_event_check["renewal_date"])
    )

    checks = [
        ("tenants_duplicate_tenant_id", int(tenants["tenant_id"].duplicated().sum())),
        ("subscriptions_duplicate_subscription_id", int(subscriptions["subscription_id"].duplicated().sum())),
        ("subscriptions_duplicate_tenant_id", int(subscriptions["tenant_id"].duplicated().sum())),
        ("users_duplicate_user_id", int(users["user_id"].duplicated().sum())),
        ("crm_companies_duplicate_company_id", int(crm_companies["company_id"].duplicated().sum())),
        ("crm_companies_duplicate_tenant_id", int(crm_companies["tenant_id"].duplicated().sum())),
        ("events_duplicate_event_id", int(events["event_id"].duplicated().sum())),
        ("crm_activities_duplicate_activity_id", int(crm_activities["activity_id"].duplicated().sum())),
        ("users_missing_tenant_fk", int((~users["tenant_id"].isin(tenants["tenant_id"])).sum())),
        ("events_missing_tenant_fk", int((~events["tenant_id"].isin(tenants["tenant_id"])).sum())),
        ("events_missing_user_fk", int((~events["user_id"].isin(users["user_id"])).sum())),
        ("crm_companies_missing_tenant_fk", int((~crm_companies["tenant_id"].isin(tenants["tenant_id"])).sum())),
        ("crm_activities_missing_tenant_fk", int((~crm_activities["tenant_id"].isin(tenants["tenant_id"])).sum())),
        ("users_event_after_churn_date", int(subscription_event_check["events_after_churn_date"].sum())),
        ("users_event_before_contract_start_date", int(subscription_event_check["events_before_contract_start_date"].sum())),
        ("users_event_after_renewal_date", int(subscription_event_check["events_after_renewal_date"].sum())),
        ("active_tenants_past_renewal_date", int(((subscriptions["churned"] == 0) & (subscriptions["renewal_date"] < SNAPSHOT_DATE)).sum())),
        ("users_registered_after_last_seen", int(user_check["registered_after_last_seen"].sum())),
        ("users_event_before_registered", int(user_check["event_before_registered"].sum())),
        ("users_event_after_last_seen", int(user_check["event_after_last_seen"].sum())),
    ]
    summary = pd.DataFrame(checks, columns=["check_name", "issue_count"])

    return summary



dq_issues = dq_summary(TABLES)

display(dq_issues.sort_values("issue_count", ascending=False))





,check_name,issue_count
19,users_event_after_last_seen,2157
16,active_tenants_past_renewal_date,219
15,users_event_after_renewal_date,214
18,users_event_before_registered,93
1,subscriptions_duplicate_subscription_id,0
17,users_registered_after_last_seen,0
14,users_event_before_contract_start_date,0
13,users_event_after_churn_date,0
12,crm_activities_missing_tenant_fk,0
11,crm_companies_missing_tenant_fk,0


The source model is structurally strong: there are no duplicate business keys and no broken foreign keys across the six core tables. 

The main issues are temporal. Product telemetry appears both before `registered_at` and after `last_seen_at` and `renewal_date`. In addition, some tenants are marked as active (`churned==0`) even though they are already past `renewal_date`.

In further analysis, I will not use `registered_at` and `last_seen_at` variables whose source is unclear, but will rely on telemetry data.

For `renewal_date`, I will derive a next renewal date for problematic rows using the fact that there is always 365 days between `contract_start_date` and `renewal_date` at the telemetry cutoff.
I also note that it is possible that the inconsistency in `renewal_date` is not a data error, but reflects continued service usage past the renewal date. However, here I assume it is a data error and attempt to correct it.



In [14]:
# Subscription cleanup: derive a next renewal date that is valid at the snapshot
def clean_subscriptions(subscriptions: pd.DataFrame, snapshot_date: pd.Timestamp = SNAPSHOT_DATE) -> pd.DataFrame:
    subscriptions_clean = subscriptions.copy()
    subscriptions_clean['next_renewal_date'] = subscriptions_clean['renewal_date']
    mask = subscriptions_clean['churn_date'].isna() & (subscriptions_clean['renewal_date'] < SNAPSHOT_DATE)
    subscriptions_clean.loc[mask, 'next_renewal_date'] = subscriptions_clean.loc[mask, 'renewal_date'] + pd.Timedelta(days=365)
    subscriptions_clean['days_to_next_renewal'] = (subscriptions_clean['next_renewal_date'] - SNAPSHOT_DATE).dt.days

    return subscriptions_clean

subscriptions_clean = clean_subscriptions(TABLES["subscriptions"])
subscriptions_clean[((subscriptions_clean["churned"] == 0) & (subscriptions_clean["renewal_date"] < SNAPSHOT_DATE))].head()

,subscription_id,tenant_id,plan,arr,contract_start_date,renewal_date,churned,churn_date,next_renewal_date,days_to_next_renewal
1,sub_ten_002,ten_002,business,13636.37,2023-01-26,2024-01-26,0,NaT,2025-01-25,209
4,sub_ten_005,ten_005,starter,2423.37,2023-05-16,2024-05-15,0,NaT,2025-05-15,319
7,sub_ten_008,ten_008,enterprise,163991.29,2023-06-08,2024-06-07,0,NaT,2025-06-07,342
8,sub_ten_009,ten_009,starter,5899.13,2023-02-06,2024-02-06,0,NaT,2025-02-05,220
11,sub_ten_012,ten_012,starter,4017.82,2023-06-24,2024-06-23,0,NaT,2025-06-23,358


In [15]:

out_data = PROJECT_ROOT / "data"
out_results = PROJECT_ROOT / "results"
out_data.mkdir(exist_ok=True)
out_results.mkdir(exist_ok=True)

subscriptions_clean.to_csv(out_data / "subscriptions_clean.csv", index=False)